In [3]:
import pandas as pd
import numpy as np

In [5]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [8]:
df = pd.read_csv('train.csv', usecols=['Age', 'Fare', 'Survived'])
df.sample()

,Survived,Age,Fare
742,1,21.0,262.375


In [9]:
df.dropna(inplace=True)

In [10]:
df.isnull().sum()

Survived    0
Age         0
Fare        0
dtype: int64

In [11]:
df.shape

(714, 3)

In [12]:
X = df.iloc[:, 1:]
y = df.iloc[:, 0]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [14]:
X_train.head()

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000
719,33.0,7.7750
666,25.0,13.0000


In [15]:
clf = DecisionTreeClassifier()

In [16]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [17]:
accuracy_score(y_test, y_pred)

0.6363636363636364

In [36]:
k_bins_age = KBinsDiscretizer(n_bins=15, strategy='uniform', encode='ordinal')
k_bis_fare = KBinsDiscretizer(n_bins=15, strategy='quantile', encode='ordinal')
k_bins_age, k_bis_fare

(KBinsDiscretizer(encode='ordinal', n_bins=15, strategy='uniform'),
 KBinsDiscretizer(encode='ordinal', n_bins=15))

In [37]:
trf = ColumnTransformer([
    ('first', k_bins_age, [0]),
    ('second', k_bis_fare, [1])
])

In [38]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:239: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(


In [39]:
trf.named_transformers_['first'].n_bins_

array([15])

In [40]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42      ,  5.72533333, 11.03066667, 16.336     , 21.64133333,
              26.94666667, 32.252     , 37.55733333, 42.86266667, 48.168     ,
              53.47333333, 58.77866667, 64.084     , 69.38933333, 74.69466667,
              80.        ])                                                   ],
      dtype=object)

In [41]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.    ,   7.25  ,   7.775 ,   7.8958,   8.1583,  10.5   ,
               13.    ,  14.4542,  18.75  ,  26.    ,  26.55  ,  31.275 ,
               51.4792,  76.2917, 108.9   , 512.3292])                   ],
      dtype=object)

In [42]:
output = pd.DataFrame({
    'age' : X_train['Age'],
    'age_trf' : X_train_trf[:,0],
    'fare' : X_train['Fare'],
    'fare_trf' : X_train_trf[:,1]
})

In [43]:
output.sample(5)

,age,age_trf,fare,fare_trf
800,34.0,6.0,13.0000,6.0
523,44.0,8.0,57.9792,12.0
343,25.0,4.0,13.0000,6.0
681,27.0,5.0,76.7292,13.0
41,27.0,5.0,21.0000,8.0


In [44]:
output['age_labels' ] = pd.cut(x=X_train['Age'],bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels' ] = pd.cut(x=X_train['Fare'],bins=trf.named_transformers_['second' ].bin_edges_[0].tolist())

In [45]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
332,38.0,7.0,153.4625,14.0,"(37.557, 42.863]","(108.9, 512.329]"
33,66.0,12.0,10.5000,5.0,"(64.084, 69.389]","(8.158, 10.5]"
138,16.0,2.0,9.2167,4.0,"(11.031, 16.336]","(8.158, 10.5]"
616,34.0,6.0,14.4000,6.0,"(32.252, 37.557]","(13.0, 14.454]"
453,49.0,9.0,89.1042,13.0,"(48.168, 53.473]","(76.292, 108.9]"


In [46]:
clf = DecisionTreeClassifier()

clf.fit(X_train_trf, y_train)

y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_pred2, y_test)

0.6363636363636364

In [47]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

/opt/conda/envs/anaconda-panel-2023.05-py310/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:239: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(


0.6289319248826291

In [52]:
def discretize(bins, strategy):
    kbins_age = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)
    kbins_fare = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)

    trf = ColumnTransformer([
        ('first', kbins_age),
        ('second', kbins_fare)
    ])

    X_trf = trf.fit_transform(X)
    print(np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy')))

    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Age'])
    plt.title('Before')

    plt.subplot(122)
    plt.hist(X_trf[:,0], color='red')
    plt.title('After')

    plt.show()
    
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Fare'])
    plt.title('Before')

    plt.subplot(122)
    plt.hist(X_trf[:,1], color='red')
    plt.title('After')

    plt.show()
discretize(20, 'uniform')

ValueError: not enough values to unpack (expected 3, got 2)

## Custom Domain based binning